In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# load data
X, y = load_iris(return_X_y=True)

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

## 1-vs-all manual implementation

In [3]:
ovr_predictions = []

for class_label in np.unique(y_train):
    # Create binary labels: current class vs all others
    y_train_binary = (y_train == class_label).astype(int)
    y_test_binary = (y_test == class_label).astype(int)
    
    # Train binary classifier
    model = LogisticRegression(solver="liblinear", max_iter=200)
    model.fit(X_train, y_train_binary)
    
    # Get probabilities for test set
    probs = model.predict_proba(X_test)[:, 1]
    ovr_predictions.append(probs)

# For each sample, pick class with highest probability
y_pred_ovr_manual = np.argmax(np.column_stack(ovr_predictions), axis=1)
print("1-vs-all accuracy:", accuracy_score(y_test, y_pred_ovr_manual))
print("1-vs-all classification report:\n", classification_report(y_test, y_pred_ovr_manual))

1-vs-all accuracy: 0.9736842105263158
1-vs-all classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      0.92      0.96        13
           2       0.92      1.00      0.96        12

    accuracy                           0.97        38
   macro avg       0.97      0.97      0.97        38
weighted avg       0.98      0.97      0.97        38



## 1-vs-1 manual implementation

In [4]:
ovo_predictions = []
class_labels = np.unique(y_train)

for i, class1 in enumerate(class_labels):
    for class2 in class_labels[i+1:]:
        # Create binary problem: class1 vs class2
        mask_train = (y_train == class1) | (y_train == class2)
        mask_test = (y_test == class1) | (y_test == class2)

        X_train_binary = X_train[mask_train]
        y_train_binary = (y_train[mask_train] == class1).astype(int)
        X_test_binary = X_test[mask_test]

        # Train binary classifier
        model = LogisticRegression(solver="liblinear", max_iter=200)
        model.fit(X_train_binary, y_train_binary)

        # Get predictions
        preds = model.predict_proba(X_test_binary)[:, 1]
        ovo_predictions.append((mask_test, class1, class2, preds))

# Accumulate votes per class for each sample
# vote_counts shape: (n_samples, n_classes), columns indexed by position in class_labels
vote_counts = np.zeros((len(y_test), len(class_labels)), dtype=int)
class_to_idx = {c: idx for idx, c in enumerate(class_labels)}

for mask, class1, class2, preds in ovo_predictions:
    # For samples in this pair's test mask, award a vote to the predicted class
    predicted_classes = np.where(preds >= 0.5, class1, class2)
    for sample_idx, pred_class in zip(np.where(mask)[0], predicted_classes):
        vote_counts[sample_idx, class_to_idx[pred_class]] += 1

# Each sample's prediction is the class with the most votes
y_pred_ovo_manual = class_labels[np.argmax(vote_counts, axis=1)]

print("1-vs-1 accuracy:", accuracy_score(y_test, y_pred_ovo_manual))
print("1-vs-1 classification report:\n", classification_report(y_test, y_pred_ovo_manual))

1-vs-1 accuracy: 1.0
1-vs-1 classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        12

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38



## Built-in support for 1-vs-all and 1-vs-1

In [5]:
# 1-vs-all using LogisticRegression's built-in support
ovr = OneVsRestClassifier(LogisticRegression(solver="liblinear", max_iter=200))
ovr.fit(X_train, y_train)
y_pred_ovr = ovr.predict(X_test)

# 1-vs-1 using OneVsOneClassifier wrapper
ovo = OneVsOneClassifier(LogisticRegression(solver="liblinear", max_iter=200))
ovo.fit(X_train, y_train)
y_pred_ovo = ovo.predict(X_test)

print("1-vs-all accuracy:", accuracy_score(y_test, y_pred_ovr))
print("1-vs-all classification report:\n", classification_report(y_test, y_pred_ovr))

print("-" * 50)

print("1-vs-1 accuracy:", accuracy_score(y_test, y_pred_ovo))
print("1-vs-1 classification report:\n", classification_report(y_test, y_pred_ovo))

1-vs-all accuracy: 0.9736842105263158
1-vs-all classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      0.92      0.96        13
           2       0.92      1.00      0.96        12

    accuracy                           0.97        38
   macro avg       0.97      0.97      0.97        38
weighted avg       0.98      0.97      0.97        38

--------------------------------------------------
1-vs-1 accuracy: 1.0
1-vs-1 classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        12

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

